In [ ]:
import numpy as np

In [ ]:
import scippneutron as scn
from scippneutron.io import sqw
import scipp as sc
from bifrost2409.config import POOCH_DATA_DIR, INTERIM_DATA_DIR

In [ ]:
experiments = [sqw.SqwIXExperiment(
    run_id=run_no,
    dpsi=sc.scalar(0., unit='deg'), 
    emode=sqw.EnergyMode.indirect, 
    efix=sc.array(values=[2.7, 3.0, 4.2, 5.0], dims=['detector'], unit='meV'), 
    en=sc.array(values=[[0.1, 1.1],[0.2, 1.2],[0.3, 1.3],[0.4, 1.4]], dims=['detector', 'energy_transfer'], unit='meV'), 
    gl=sc.scalar(0., unit='deg'), 
    gs=sc.scalar(0., unit='deg'), 
    omega=sc.scalar(0., unit='deg'), 
    psi=sc.scalar(psi, unit='deg'), 
    u=sc.vector([0, 0, 1], unit='1/angstrom'), 
    v=sc.vector([1, 0, 0], unit='1/angstrom')) for run_no, psi in enumerate(range(1))]

In [ ]:
instr = sqw.SqwIXNullInstrument(name='NotAnInstrument', source=sqw.SqwIXSource(name='ESS', target_name='Tungsten', frequency=sc.scalar(14.0, unit='Hz')))

In [ ]:
sample = sqw.SqwIXSample(name='reciprocal space', lattice_spacing=sc.vector([2, 2, 2], unit='1/angstrom') * np.pi, lattice_angle=sc.vector([90, 90, 90], unit='deg'))

In [ ]:
axes = sqw.SqwLineAxes(title='title!',
    label=['Q_x', 'Q_y', 'Q_z', 'En'],
    img_scales=[sc.scalar(1, unit='1/angstrom'), sc.scalar(1, unit='1/angstrom'), sc.scalar(1, unit='1/angstrom'), sc.scalar(1., unit='meV')],
    img_range=[sc.array(values=[-1,1],dims=['Q_x'],unit='1/angstrom'), sc.array(values=[-1,1],dims=['Q_y'],unit='1/angstrom'),sc.array(values=[-1,1],dims=['Q_z'],unit='1/angstrom'), sc.array(values=[0,10], unit='meV', dims=['En'])],
    n_bins_all_dims = sc.array(values=[50, 50, 50, 50], unit=None, dims=['bin_dim']),
    single_bin_defines_iax = sc.array(values=[False, False, False, False], unit=None, dims=['bin_dim']),
    dax = sc.array(values=[0, 1, 2, 3], unit=None, dims=['bin_dim']),
    offset=[sc.scalar(0, unit='1/angstrom'), sc.scalar(0, unit='1/angstrom'), sc.scalar(0, unit='1/angstrom'), sc.scalar(0, unit='meV')],
    changes_aspect_ratio=False,
    )
proj = sqw.SqwLineProj(
    lattice_spacing= sample.lattice_spacing,
    lattice_angle= sample.lattice_angle,
    offset= axes.offset,
    title= 'cartesian axes projection',
    label=axes.label,
    u= sc.vector([0, 0, 1], unit='1/angstrom'),
    v= sc.vector([1, 0, 0], unit='1/angstrom'),
    w=None,
    non_orthogonal=False,
    type='aaa',
)
dnd = sqw.SqwDndMetadata(axes=axes, proj=proj)

In [ ]:
obj = (sqw.Sqw.build(INTERIM_DATA_DIR / 'test_sqw_obj.sqw')
       .register_pixel_data(
           experiments=experiments, n_dims=4, n_pixels=8, )
       .add_default_instrument(instr)
       .add_default_sample(sample)
       .add_empty_detector_params()
       .add_empty_dnd_data(dnd)
      )
with obj.create() as sqw_file_obj:
    for run_id, experiment in enumerate(experiments):
        # ('u1', 'u2', 'u3', 'u4', 'irun', 'idet', 'ien', 'signal', 'error'),
        en = experiment.en.values.flatten()
        qx, qy, qz = np.random.rand(*en.shape), np.zeros(en.shape), np.random.rand(*en.shape)
        irun = run_id * np.ones(en.shape)
        idet = sc.arange('detector', experiment.en.sizes['detector']).broadcast(shape=experiment.en.shape, dims=experiment.en.dims).values.flatten()
        ien = sc.arange('energy_transfer', experiment.en.sizes['energy_transfer']).broadcast(shape=experiment.en.shape, dims=experiment.en.dims).values.flatten()
        signal = np.random.rand(*en.shape) * 100
        error = signal
        data = np.vstack([qx, qy, qz, en, irun, idet, ien, signal, error]).T
        sqw_file_obj.write_pixel_data(data, run_id)


In [ ]:
pwd()

In [ ]:
data = sc.data.binned_xy(1000,10,20)

In [ ]:
data.bins.constituents

In [ ]:
data.bins.nansum()

In [ ]:
data.bins.constituents['data']